# Hedge Fund Industry X-Ray
## A 9-Source Reconstruction of U.S. Hedge Fund Balance Sheets, Derivatives & Broker Infrastructure (2012–2026)

**Data Sources:**
1. **Federal Reserve Z.1** — Table B.101.f, quarterly balance sheet of domestic hedge funds (29 series)
2. **SEC Form PF** — Fund-level GAV/NAV, strategy allocation, liquidity profiles (~1,500 large hedge funds)
3. **CFTC Weekly Swaps** — OTC swap notional outstanding by asset class and clearing status
4. **SEC EDGAR 13F** — Institutional equity holdings for 8 major hedge funds
5. **SEC EDGAR Submissions** — Filing history and fund metadata
6. **CFTC Commitments of Traders** — Leveraged fund positioning in futures markets
7. **CBOE VIX** — Market volatility index (daily → quarterly)
8. **DTCC Swap Repository** — Trade-level OTC derivatives data
9. **CFTC FCM Financials** — Futures broker capital adequacy and market concentration

**What makes this different:** Cross-source reconciliation — comparing the Fed's aggregate leverage ratio with Form PF's fund-level GAV/NAV, overlaying on-balance-sheet derivatives with OTC swap notionals, and connecting broker capital adequacy to clearing trends.

## 1. Setup & Imports

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.analysis.metrics import compute_derived_metrics
from src.data.fetch import HEDGE_FUND_CIKS, load_best_13f_holdings
from src.data.prepare import align_vix_to_fred, load_fred_balance_sheet

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

fmt_billions = FuncFormatter(lambda x, _: f"${x:,.0f}B")
fmt_pct = FuncFormatter(lambda x, _: f"{x:.0f}%")
fmt_ratio = FuncFormatter(lambda x, _: f"{x:.2f}x")


def polish(ax, ylabel_fmt=None, date_axis=True):
    """Apply standard formatting: remove spines, set ticks, format axes."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=10)
    if ylabel_fmt:
        ax.yaxis.set_major_formatter(ylabel_fmt)
    if date_axis:
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")


def merge_legends(ax1, ax2, **kwargs):
    """Combine legends from dual-axis chart onto ax1."""
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    kw = {"loc": "upper left", "framealpha": 0.9, "edgecolor": "gray", "fontsize": 10}
    kw.update(kwargs)
    ax1.legend(lines1 + lines2, labels1 + labels2, **kw)
    if ax2.get_legend():
        ax2.get_legend().remove()


RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
FIGURES_DIR = ROOT_DIR / "outputs" / "figures"
REPORTS_DIR = ROOT_DIR / "outputs" / "reports"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook mode: tracked processed snapshot first, local raw cache fallback second")
print(f"Processed snapshot: {PROCESSED_DIR}")
print(f"Raw cache present: {'Yes' if RAW_DIR.exists() else 'No'}")

Notebook mode: tracked processed snapshot first, local raw cache fallback second
Processed snapshot: /Users/christopherortiz/Desktop/GitHub Repositories/financial_data/data/processed
Raw cache present: Yes


## 2. Data Snapshot

### 2A. Federal Reserve Z.1 — Tracked Snapshot

This executed notebook is **snapshot-first**. It reads the tracked processed datasets in `data/processed/` and only falls back to safe local caches in `data/raw/` when needed. Live fetches are handled separately by `python -m src.pipeline --fetch`.

In [2]:
balance_sheet_snapshot = PROCESSED_DIR / "hedge_fund_analysis.csv"
raw_balance_sheet_cache = RAW_DIR / "hedge_fund_balance_sheet_fred.csv"

print(f"Processed balance-sheet snapshot present: {balance_sheet_snapshot.exists()}")
print(f"Raw FRED cache present: {raw_balance_sheet_cache.exists()}")

Processed balance-sheet snapshot present: True
Raw FRED cache present: True


In [3]:
if balance_sheet_snapshot.exists():
    df_hf = pd.read_csv(balance_sheet_snapshot, index_col=0, parse_dates=True).sort_index()
    balance_sheet_source = balance_sheet_snapshot.relative_to(ROOT_DIR).as_posix()
elif raw_balance_sheet_cache.exists():
    df_hf = load_fred_balance_sheet(raw_balance_sheet_cache)
    balance_sheet_source = raw_balance_sheet_cache.relative_to(ROOT_DIR).as_posix()
else:
    raise FileNotFoundError("No hedge fund balance-sheet snapshot found. Run `python -m src.pipeline --analyze` first.")

print(f"Loaded hedge fund balance-sheet data from {balance_sheet_source}")
print(f"Coverage: {len(df_hf)} quarters, {df_hf.index.min().date()} to {df_hf.index.max().date()}")
df_hf.head()

Loaded hedge fund balance-sheet data from data/processed/hedge_fund_analysis.csv
Coverage: 52 quarters, 2012-10-01 to 2025-07-01


,Total assets,Foreign currency; asset,Deposits; asset,Other cash and cash equivalents; asset,Money market fund shares; asset,Security repurchase agreements; asset,Total debt securities; asset,Treasury securities; asset,Corporate and foreign bonds; asset,Total loans; asset,...,other_secured_pct,unsecured_pct,domestic_borrowing,foreign_borrowing,foreign_borrowing_share,total_assets_qoq,total_assets_yoy,net_assets_qoq,liabilities_qoq,leverage_change
Date,,,,,,,,,,,,,,,,,,,,,
2012-10-01,1144.799,11.373,28.729,73.723,27.886,34.955,389.090,119.819,256.091,50.830,...,0.196727,0.006222,205.352,35.799,0.148451,NaN,NaN,NaN,NaN,NaN
2013-01-01,1281.887,15.823,26.778,73.256,26.678,54.039,449.813,143.146,293.889,51.072,...,0.193039,0.005084,243.595,62.569,0.204364,0.119749,NaN,0.074515,0.218190,0.054185
2013-04-01,1317.475,16.148,31.194,74.655,24.142,59.357,459.365,153.313,290.133,51.908,...,0.176221,0.008928,246.411,66.537,0.212614,0.027762,NaN,0.021153,0.047374,0.011797
2013-07-01,1355.993,14.994,37.836,75.849,27.414,55.204,452.864,145.406,293.536,51.161,...,0.177755,0.009334,258.655,62.924,0.195672,0.029236,NaN,0.042416,0.004957,-0.016933
2013-10-01,1591.137,17.689,39.878,95.725,63.921,57.847,535.920,157.241,357.986,55.233,...,0.177161,0.009973,281.945,60.074,0.175645,0.173411,0.389883,0.205075,0.098371,-0.040225


### 2B. VIX Volatility Index — Snapshot Load

The VIX series is bundled as a quarterly snapshot for reproducible analysis. The notebook loads the tracked snapshot first and only falls back to the local raw cache if the processed copy is unavailable.

In [4]:
vix_snapshot = PROCESSED_DIR / "vix_quarterly.csv"
raw_vix_cache = RAW_DIR / "vix_quarterly.csv"

if vix_snapshot.exists():
    df_vix = pd.read_csv(vix_snapshot, index_col=0, parse_dates=True).sort_index()
    vix_source = vix_snapshot.relative_to(ROOT_DIR).as_posix()
elif raw_vix_cache.exists():
    df_vix = pd.read_csv(raw_vix_cache, index_col=0, parse_dates=True).sort_index()
    vix_source = raw_vix_cache.relative_to(ROOT_DIR).as_posix()
else:
    df_vix = pd.DataFrame()
    vix_source = None

if not df_vix.empty:
    print(f"Loaded VIX snapshot from {vix_source}")
    print(f"VIX coverage: {len(df_vix)} quarters, {df_vix.index.min().date()} to {df_vix.index.max().date()}")
    display(df_vix.tail())
else:
    print("No VIX snapshot available")

Loaded VIX snapshot from data/processed/vix_quarterly.csv
VIX coverage: 145 quarters, 1990-03-31 to 2026-03-31


,VIX_mean,VIX_max,VIX_min,VIX_end,VIX_std
Date,,,,,
2025-03-31,18.521111,27.86,14.77,22.28,3.180515
2025-06-30,23.561406,52.33,16.32,16.73,7.819223
2025-09-30,15.983030,20.38,14.22,16.28,1.054921
2025-12-31,17.745231,26.42,13.47,14.95,2.795989
2026-03-31,19.408545,29.49,14.49,24.06,3.698140


### 2C. SEC EDGAR 13F Filings — Hedge Fund Holdings

13F filings reveal what institutional investors hold on the long side. This notebook loads the best locally available amendment-deduped 13F window for eight major hedge funds; in the bundled repo cache that is currently report periods **2024Q1–2025Q4**.


In [5]:
print("Loading amendment-deduped 13F holdings window for major hedge funds...\n")
df_13f = load_best_13f_holdings(str(RAW_DIR), expected_funds=HEDGE_FUND_CIKS)

if not df_13f.empty:
    periods = sorted(df_13f["report_period"].dropna().astype(str).unique()) if "report_period" in df_13f.columns else []
    non_option = df_13f[df_13f["put_call"].fillna("").eq("")]
    if periods:
        print(f"13F report periods: {periods[0]} to {periods[-1]}")
    print(f"Total 13F rows: {len(df_13f):,} across {df_13f['fund'].nunique()} funds")
    print(f"Long equity/ETF positions: {len(non_option):,}")
else:
    print("No 13F holdings data available")

Loading amendment-deduped 13F holdings window for major hedge funds...



13F report periods: 2024Q1 to 2025Q4
Total 13F rows: 384,723 across 8 funds
Long equity/ETF positions: 283,362


### 2D. CFTC Commitments of Traders — Snapshot Load

The leveraged-fund positioning series is loaded from the tracked processed snapshot (`data/processed/cftc_cot.csv`) whenever available. This keeps the default notebook run offline-safe and reproducible.

In [6]:
cftc_snapshot = PROCESSED_DIR / "cftc_cot.csv"
raw_cftc_cache = RAW_DIR / "cftc_cot.csv"

if cftc_snapshot.exists():
    df_cftc = pd.read_csv(cftc_snapshot, parse_dates=["date"])
    cftc_source = cftc_snapshot.relative_to(ROOT_DIR).as_posix()
elif raw_cftc_cache.exists():
    df_cftc = pd.read_csv(raw_cftc_cache, parse_dates=["date"])
    cftc_source = raw_cftc_cache.relative_to(ROOT_DIR).as_posix()
else:
    df_cftc = pd.DataFrame()
    cftc_source = None

if not df_cftc.empty:
    print(f"Loaded CFTC positioning snapshot from {cftc_source}")
    print(f"CFTC coverage: {len(df_cftc)} rows, {df_cftc['date'].min().date()} to {df_cftc['date'].max().date()}")
    print(f"Columns: {list(df_cftc.columns)}")
    display(df_cftc.tail())
else:
    print("No CFTC positioning snapshot available")

Loaded CFTC positioning snapshot from data/processed/cftc_cot.csv
CFTC coverage: 7573 rows, 2018-01-02 to 2025-12-30
Columns: ['date', 'market', 'lev_fund_long', 'lev_fund_short', 'lev_fund_spreading', 'lev_fund_net']


,date,market,lev_fund_long,lev_fund_short,lev_fund_spreading,lev_fund_net
7568,2025-12-30,E-MINI S&P COMMUNICATION INDEX - CHICAGO MERCA...,400,1899,0,-1499
7569,2025-12-30,DOW JONES U.S. REAL ESTATE IDX - CHICAGO BOARD...,4578,11435,0,-6857
7570,2025-12-30,E-MINI S&P HEALTH CARE INDEX - CHICAGO MERCANT...,877,0,0,877
7571,2025-12-30,E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTIL...,2693,445,0,2248
7572,2025-12-30,NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE,59039,84713,1970,-25674


## 3. Data Preparation

Parse dates, compute derived metrics (leverage ratios, allocation percentages, growth rates), and merge datasets.

In [7]:
# Validate FRED data shape
print(f"FRED data: {len(df_hf)} quarters")
print(f"Date range: {df_hf.index.min().date()} to {df_hf.index.max().date()}")
print(f"Columns: {len(df_hf.columns)}")
print(f"\nNull values: {df_hf.isnull().sum().sum()}")
df_hf.tail()

FRED data: 52 quarters
Date range: 2012-10-01 to 2025-07-01
Columns: 51

Null values: 8


,Total assets,Foreign currency; asset,Deposits; asset,Other cash and cash equivalents; asset,Money market fund shares; asset,Security repurchase agreements; asset,Total debt securities; asset,Treasury securities; asset,Corporate and foreign bonds; asset,Total loans; asset,...,other_secured_pct,unsecured_pct,domestic_borrowing,foreign_borrowing,foreign_borrowing_share,total_assets_qoq,total_assets_yoy,net_assets_qoq,liabilities_qoq,leverage_change
Date,,,,,,,,,,,,,,,,,,,,,
2024-07-01,2819.616,23.331,37.955,97.723,68.140,101.336,787.511,275.632,486.384,246.764,...,0.215218,0.026807,647.053,112.388,0.147988,0.058691,0.141278,0.042642,0.087744,0.018174
2024-10-01,2900.447,21.812,42.440,97.647,88.895,101.600,783.510,277.740,483.869,267.571,...,0.217737,0.019402,663.018,119.977,0.153228,0.028667,0.111044,0.015228,0.063398,0.020797
2025-01-01,2903.615,28.597,45.642,112.531,77.002,99.863,777.862,270.426,483.279,280.607,...,0.220339,0.022328,697.167,114.133,0.140679,0.001092,0.080073,0.012657,-0.014370,-0.012253
2025-04-01,3065.863,20.967,42.089,100.902,72.612,112.858,807.275,287.058,497.282,290.575,...,0.202853,0.021234,762.334,131.578,0.147193,0.055878,0.151151,0.044867,0.071443,0.011365
2025-07-01,3258.408,22.189,42.933,100.059,79.490,117.405,846.779,294.907,527.836,295.963,...,0.189840,0.020090,851.598,173.001,0.168848,0.062803,0.155621,0.041391,0.101859,0.026606


In [8]:
required_metrics = {"leverage_ratio", "cash_to_assets", "equity_pct", "derivative_to_assets", "foreign_borrowing_share"}

if required_metrics.issubset(df_hf.columns):
    df = df_hf.copy()
    print("Processed snapshot already includes derived metrics")
else:
    df = compute_derived_metrics(df_hf)
    print("Derived metrics recomputed from raw balance-sheet cache")

print(f"Dataset with derived metrics: {df.shape}")
print("\nKey ratios (latest quarter):")
latest = df.iloc[-1]
print(f"  Leverage ratio:        {latest['leverage_ratio']:.2f}x")
print(f"  Cash-to-assets:        {latest['cash_to_assets']:.1%}")
print(f"  Equity allocation:     {latest['equity_pct']:.1%}")
print(f"  Derivative/assets:     {latest['derivative_to_assets']:.1%}")
print(f"  Foreign borrowing:     {latest['foreign_borrowing_share']:.1%}")

Processed snapshot already includes derived metrics
Dataset with derived metrics: (52, 51)

Key ratios (latest quarter):
  Leverage ratio:        0.48x
  Cash-to-assets:        6.8%
  Equity allocation:     42.0%
  Derivative/assets:     41.1%
  Foreign borrowing:     16.9%


In [9]:
vix_columns = {"VIX_mean", "VIX_max", "VIX_min", "VIX_end", "VIX_std"}

if vix_columns.issubset(df.columns):
    df_merged = df.copy()
    print("Processed snapshot already includes VIX alignment")
elif not df_vix.empty:
    df_merged = align_vix_to_fred(df, df_vix)
    print(f"Merged dataset: {df_merged.shape}")
    print(f"VIX coverage: {df_merged['VIX_mean'].notna().sum()} of {len(df_merged)} quarters")
else:
    df_merged = df.copy()
    print("No VIX data to merge")

df_merged.info()

Processed snapshot already includes VIX alignment
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 52 entries, 2012-10-01 to 2025-07-01
Data columns (total 51 columns):
 #   Column                                                                Non-Null Count  Dtype  
---  ------                                                                --------------  -----  
 0   Total assets                                                          52 non-null     float64
 1   Foreign currency; asset                                               52 non-null     float64
 2   Deposits; asset                                                       52 non-null     float64
 3   Other cash and cash equivalents; asset                                52 non-null     float64
 4   Money market fund shares; asset                                       52 non-null     float64
 5   Security repurchase agreements; asset                                 52 non-null     float64
 6   Total debt securities; asset  

## 4. Exploratory Data Analysis

In [10]:
# Summary statistics for key balance sheet items
key_cols = [
    "Total assets",
    "Total liabilities",
    "Total net assets",
    "Corporate equities; asset",
    "Total debt securities; asset",
    "Derivatives (long value)",
    "leverage_ratio",
]
df[key_cols].describe().round(2)

,Total assets,Total liabilities,Total net assets,Corporate equities; asset,Total debt securities; asset,Derivatives (long value),leverage_ratio
count,52.00,52.00,52.00,52.00,52.00,52.00,52.00
mean,2195.08,687.63,1601.56,836.17,660.10,1186.99,0.43
std,456.42,156.41,341.54,230.21,91.00,392.83,0.02
min,1144.80,339.59,837.99,388.55,389.09,912.03,0.40
25%,1893.60,590.70,1362.71,634.19,631.05,1025.54,0.41
50%,2263.27,704.93,1635.88,848.85,674.41,1088.92,0.43
75%,2465.73,762.15,1860.94,1004.35,716.38,1217.77,0.44
max,3258.41,1113.48,2296.69,1368.21,846.78,3447.80,0.48


In [11]:
# Correlation matrix of major balance sheet components
from src.visualization.plots import plot_correlation_heatmap

plot_correlation_heatmap(df, save_path="../outputs/figures/correlation_heatmap.png")

## 5. Visualizations

### 5A. Total Assets Over Time

In [12]:
from src.visualization.plots import (
    add_event_annotations,
    plot_asset_composition,
    plot_balance_sheet_overview,
    plot_borrowing_patterns,
    plot_correlation_heatmap,
    plot_debt_securities,
    plot_derivative_exposure,
    plot_liability_structure,
    plot_total_assets,
)

plot_total_assets(df, save_path=str(FIGURES_DIR / "total_assets.png"))

### 5B. Asset Composition — What Do Hedge Funds Hold?

In [13]:
# 5B: Stacked area chart — asset composition
plot_asset_composition(df, save_path=f"{FIGURES_DIR}/asset_composition.png")

### 5C. Debt Securities Deep Dive — Treasury vs. Corporate Bonds

In [14]:
# 5C: Debt securities breakdown
plot_debt_securities(df, save_path=f"{FIGURES_DIR}/debt_securities.png")

### 5D. Liability Structure & Leverage Ratio

In [15]:
# 5D: Liability structure stacked area + leverage ratio line
plot_liability_structure(df, save_path=f"{FIGURES_DIR}/liability_structure.png")

### 5E. Assets vs. Liabilities vs. Net Assets

In [16]:
# 5E: Three-line overlay — assets, liabilities, net assets
plot_balance_sheet_overview(df, save_path=f"{FIGURES_DIR}/balance_sheet_overview.png")

### 5F. Derivative Exposure Trends

The Q1 2018 spike in derivatives aligns with "Volmageddon" — the February 2018 VIX blowup that caused massive losses in short-volatility strategies.

In [17]:
# 5F: Derivative exposure
plot_derivative_exposure(df, save_path=f"{FIGURES_DIR}/derivative_exposure.png")

### 5G. Domestic vs. Foreign Borrowing Patterns

In [18]:
# 5G: Domestic vs foreign borrowing
plot_borrowing_patterns(df, save_path=f"{FIGURES_DIR}/borrowing_patterns.png")

### 5H. Key Ratios Dashboard

In [19]:
# 5H: Key ratios dashboard — 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ratios = [
    ("leverage_ratio", "Leverage Ratio (Liabilities/Net Assets)", "#c0392b", "x"),
    ("cash_to_assets", "Cash-to-Assets Ratio", "#27ae60", "%"),
    ("equity_pct", "Equity Allocation (% of Assets)", "#2980b9", "%"),
    ("derivative_to_assets", "Derivatives / Total Assets", "#8e44ad", "%"),
]

for ax, (col, title, color, fmt) in zip(axes.flat, ratios):
    vals = df[col] * 100 if fmt == "%" else df[col]
    rolling = vals.rolling(4, min_periods=1).mean()

    ax.plot(df.index, vals, linewidth=1, alpha=0.5, color=color)
    ax.plot(df.index, rolling, linewidth=2.5, color=color, label="4Q Rolling Avg")
    ax.fill_between(df.index, vals, alpha=0.08, color=color)

    mean_val = vals.mean()
    suffix = "%" if fmt == "%" else "x"
    ax.axhline(mean_val, color="gray", linestyle="--", alpha=0.6, label=f"Mean: {mean_val:.1f}{suffix}")

    ax.set_title(title)
    ylabel_formatter = fmt_pct if fmt == "%" else fmt_ratio
    polish(ax, ylabel_fmt=ylabel_formatter)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor="gray")
    add_event_annotations(ax, ypos_frac=0.92)

plt.suptitle("Hedge Fund Industry — Key Financial Ratios", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 5I. 13F Holdings — What Did Major Hedge Funds Hold?

> **Note:** 13F filings only disclose **long equity positions**. Short positions, options strategies, and derivatives exposure are not reported. This means 13F data cannot capture the short-side dynamics that drove events like the GameStop squeeze.

In [20]:
# 5I: 13F holdings analysis — single-quarter long-book snapshot
if not df_13f.empty and "value_usd" in df_13f.columns:
    latest_q = (
        df_13f["report_period"].dropna().astype(str).max() if "report_period" in df_13f.columns else "all filings"
    )
    df_latest = (
        df_13f[df_13f["report_period"].astype(str) == latest_q].copy()
        if "report_period" in df_13f.columns
        else df_13f.copy()
    )
    df_equity = df_latest[df_latest["put_call"].fillna("").eq("")].copy()

    print(
        f"13F snapshot: {latest_q} ({len(df_equity)} long equity/ETF positions across {df_equity['fund'].nunique()} funds)"
    )
    print(f"  (Options and other put/call rows excluded; {len(df_latest) - len(df_equity)} non-equity rows omitted)")

    top_holdings = df_equity.groupby("issuer")["value_usd"].sum().sort_values(ascending=False).head(20)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    top_holdings_b = top_holdings / 1e9
    ax1.barh(range(len(top_holdings_b)), top_holdings_b.values, color="#3498db", alpha=0.8)
    ax1.set_yticks(range(len(top_holdings_b)))
    ax1.set_yticklabels(top_holdings_b.index, fontsize=9)
    ax1.set_xlabel("Total Value ($B)")
    ax1.set_title(f"Top 20 Long Holdings — {latest_q} Snapshot")
    ax1.invert_yaxis()

    fund_totals = df_equity.groupby("fund")["value_usd"].sum().sort_values(ascending=True) / 1e9
    ax2.barh(range(len(fund_totals)), fund_totals.values, color="#e74c3c", alpha=0.8)
    ax2.set_yticks(range(len(fund_totals)))
    ax2.set_yticklabels(fund_totals.index, fontsize=10)
    ax2.set_xlabel("Total Long Equity Value ($B)")
    ax2.set_title(f"Long Portfolio Size by Fund — {latest_q} Snapshot")

    plt.suptitle(f"SEC 13F Filings — Amendment-Deduped Long Snapshot ({latest_q})", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    nvidia = df_equity[df_equity["issuer"].fillna("").str.contains("NVIDIA", case=False, na=False)]
    if not nvidia.empty:
        print(
            f"\nNVIDIA long value in {latest_q}: ${nvidia['value_usd'].sum() / 1e9:,.2f}B across {nvidia['fund'].nunique()} funds"
        )
else:
    print("No 13F holdings data available — skipping visualization")

13F snapshot: 2025Q4 (43482 long equity/ETF positions across 8 funds)
  (Options and other put/call rows excluded; 12772 non-equity rows omitted)



NVIDIA long value in 2025Q4: $19.07B across 8 funds


### 5J. CFTC — Leveraged Fund Positioning in Equity Futures

In [21]:
# 5J: CFTC leveraged fund positioning
if not df_cftc.empty and "lev_fund_net" in df_cftc.columns:
    # Aggregate across equity index futures by date
    cftc_agg = (
        df_cftc.groupby("date")
        .agg(
            {
                "lev_fund_long": "sum",
                "lev_fund_short": "sum",
                "lev_fund_net": "sum",
            }
        )
        .sort_index()
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    # Long vs Short positions
    ax1.plot(cftc_agg.index, cftc_agg["lev_fund_long"], label="Long Positions", color="#27ae60", linewidth=1.5)
    ax1.plot(cftc_agg.index, cftc_agg["lev_fund_short"], label="Short Positions", color="#e74c3c", linewidth=1.5)
    ax1.fill_between(
        cftc_agg.index,
        cftc_agg["lev_fund_long"],
        cftc_agg["lev_fund_short"],
        where=cftc_agg["lev_fund_long"] >= cftc_agg["lev_fund_short"],
        alpha=0.2,
        color="#27ae60",
        label="Net Long",
    )
    ax1.fill_between(
        cftc_agg.index,
        cftc_agg["lev_fund_long"],
        cftc_agg["lev_fund_short"],
        where=cftc_agg["lev_fund_long"] < cftc_agg["lev_fund_short"],
        alpha=0.2,
        color="#e74c3c",
        label="Net Short",
    )
    ax1.set_title("Leveraged Fund Equity Futures Positioning (CFTC COT)")
    ax1.set_ylabel("Contracts")
    ax1.legend(fontsize=9, framealpha=0.9, edgecolor="gray")
    polish(ax1, date_axis=True)

    # Net positioning
    colors = ["#27ae60" if x >= 0 else "#e74c3c" for x in cftc_agg["lev_fund_net"]]
    ax2.bar(cftc_agg.index, cftc_agg["lev_fund_net"], width=5, color=colors, alpha=0.7)
    ax2.axhline(0, color="black", linewidth=0.5)
    ax2.set_title("Net Positioning (Long - Short)")
    ax2.set_ylabel("Net Contracts")
    polish(ax2, date_axis=True)

    plt.tight_layout()
    plt.show()
else:
    print("No CFTC data available — skipping visualization")

### 5K. VIX vs. Hedge Fund Leverage — Does Volatility Drive Deleveraging?

In [22]:
# 5K: VIX vs leverage ratio
if "VIX_mean" in df_merged.columns and df_merged["VIX_mean"].notna().any():
    fig, ax1 = plt.subplots(figsize=(14, 6))

    ax1.plot(df_merged.index, df_merged["leverage_ratio"], linewidth=2.5, color="#c0392b", label="Leverage Ratio")
    ax1.set_ylabel("Leverage Ratio")

    ax2 = ax1.twinx()
    ax2.plot(
        df_merged.index, df_merged["VIX_mean"], linewidth=2, color="#8e44ad", alpha=0.7, label="VIX (Quarterly Mean)"
    )
    ax2.fill_between(
        df_merged.index, df_merged["VIX_min"], df_merged["VIX_max"], alpha=0.1, color="#8e44ad", label="VIX Range"
    )
    ax2.set_ylabel("VIX")

    # Correlation
    valid = df_merged[["leverage_ratio", "VIX_mean"]].dropna()
    corr = valid["leverage_ratio"].corr(valid["VIX_mean"])
    ax1.annotate(
        f"Correlation: {corr:.2f}",
        xy=(0.02, 0.05),
        xycoords="axes fraction",
        fontsize=12,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )

    ax1.set_title("Hedge Fund Leverage vs. Market Volatility (VIX)")
    add_event_annotations(ax1)
    polish(ax1, ylabel_fmt=fmt_ratio)
    ax2.spines[["top"]].set_visible(False)
    merge_legends(ax1, ax2)
    plt.tight_layout()
    plt.show()
else:
    print("No VIX data available for this visualization")

### 5L. Form PF — Fund-Level Leverage & Strategy Allocation

SEC Form PF (Private Fund reporting) provides fund-level data that the aggregate Z.1 balance sheet cannot: **Gross Asset Value (GAV)**, **Net Asset Value (NAV)**, strategy breakdowns, and liquidity profiles. Quarterly since 2013, covering ~1,500 large hedge funds.

The GAV/NAV ratio is a true leverage proxy — a ratio of 2.0x means the fund controls $2 in gross assets for every $1 of investor equity.

In [23]:
from src.visualization.plots import plot_form_pf_leverage, plot_strategy_allocation

pf_gav = pd.read_csv(PROCESSED_DIR / "form_pf_gav_nav.csv")
pf_hf = pf_gav[pf_gav["fund_type"] == "Hedge Fund"].copy()
print(f"Form PF hedge fund data: {len(pf_hf)} quarters")
print(f"  GAV range: ${pf_hf['gav'].min():,.0f}B - ${pf_hf['gav'].max():,.0f}B")
print(f"  Latest GAV/NAV ratio: {pf_hf['gav'].iloc[-1] / pf_hf['nav'].iloc[-1]:.2f}x")

plot_form_pf_leverage(pf_hf, z1_df=df, save_path=str(FIGURES_DIR / "form_pf_leverage.png"))

pf_strat = pd.read_csv(PROCESSED_DIR / "form_pf_strategy.csv")
plot_strategy_allocation(pf_strat, save_path=str(FIGURES_DIR / "strategy_allocation.png"))

Form PF hedge fund data: 49 quarters
  GAV range: $4,780B - $12,590B
  Latest GAV/NAV ratio: 2.32x


### 5M. CFTC Weekly Swaps — OTC Derivatives Market

The CFTC publishes weekly reports on OTC swap notional outstanding by asset class (interest rate, credit, FX, equity, commodity) and clearing status. This reveals the **scale of off-balance-sheet derivatives** that Z.1 only partially captures through "Derivatives (long value)."

In [24]:
from src.visualization.plots import plot_clearing_rate, plot_swaps_notional

swaps_df = pd.read_csv(PROCESSED_DIR / "swaps_weekly.csv")
swaps_df["date"] = pd.to_datetime(swaps_df["date"])
print(
    f"CFTC swaps data: {len(swaps_df)} weekly observations, {swaps_df['date'].min().date()} to {swaps_df['date'].max().date()}"
)

plot_swaps_notional(swaps_df, save_path=str(FIGURES_DIR / "swaps_notional.png"))
plot_clearing_rate(swaps_df, save_path=str(FIGURES_DIR / "clearing_rate.png"))

CFTC swaps data: 605 weekly observations, 2012-12-07 to 2026-02-20


### 5N. FCM Broker Financials — Capital Adequacy & Concentration

Futures Commission Merchants (FCMs) are the brokers that clear hedge fund derivatives trades. Their monthly CFTC financial reports reveal **capital adequacy**, **customer segregated funds**, and **market concentration** — the plumbing that breaks during crises.

In [25]:
from src.visualization.plots import plot_fcm_capital, plot_fcm_concentration

fcm_industry = pd.read_csv(PROCESSED_DIR / "fcm_monthly_industry.csv")
date_col = "date" if "date" in fcm_industry.columns else "as_of_date"
fcm_industry[date_col] = pd.to_datetime(fcm_industry[date_col])
print(
    f"FCM industry data: {len(fcm_industry)} monthly observations, {fcm_industry[date_col].min().date()} to {fcm_industry[date_col].max().date()}"
)

plot_fcm_capital(fcm_industry, save_path=str(FIGURES_DIR / "fcm_capital.png"))

fcm_conc = pd.read_csv(PROCESSED_DIR / "fcm_concentration.csv")
plot_fcm_concentration(fcm_conc, save_path=str(FIGURES_DIR / "fcm_concentration.png"))

FCM industry data: 49 monthly observations, 2022-01-31 to 2026-01-31


### 5O. DTCC Swap Repository — Trade-Level OTC Data

The DTCC's Swap Data Repository captures **individual OTC derivative transactions** across 5 asset classes (Rates, Credits, Equities, Forex, Commodities). This trade-level data reveals clearing rates, prime brokerage involvement, and block trade activity that aggregate reports cannot show.

In [26]:
from src.visualization.plots import plot_dtcc_summary

dtcc_q = pd.read_csv(PROCESSED_DIR / "dtcc_quarterly.csv")
print(f"DTCC quarterly data: {len(dtcc_q)} rows across {dtcc_q['asset_class'].nunique()} asset classes")
print(f"  Quarters covered: {sorted(dtcc_q['quarter'].unique())}")

plot_dtcc_summary(dtcc_q, save_path=str(FIGURES_DIR / "dtcc_summary.png"))

pb_col = "quarter_end_pb_pct" if "quarter_end_pb_pct" in dtcc_q.columns else "avg_pb_pct"
print("\nPrime Brokerage Involvement (avg across asset classes):")
for ac in sorted(dtcc_q["asset_class"].unique()):
    ac_data = dtcc_q[dtcc_q["asset_class"] == ac]
    print(f"  {ac:15s}: {ac_data[pb_col].mean():.1%} of trades")

DTCC quarterly data: 25 rows across 5 asset classes
  Quarters covered: ['2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']



Prime Brokerage Involvement (avg across asset classes):
  COMMODITIES    : 1.0% of trades
  CREDITS        : 0.0% of trades
  EQUITIES       : 0.0% of trades
  FOREX          : 40.4% of trades
  RATES          : 0.4% of trades


### 5O. Cross-Source Reconciliation — Z.1 vs Form PF Leverage

This is the chart that justifies combining multiple data sources. The Fed's Z.1 **leverage ratio** (liabilities / net assets) and Form PF's **GAV/NAV ratio** (gross assets / net assets) measure leverage through different lenses — one from the liability side, one from the asset side. When they diverge, it reveals off-balance-sheet exposure or measurement gaps.

In [27]:
from src.visualization.plots import plot_cross_source_leverage

plot_cross_source_leverage(df, pf_hf, save_path=str(FIGURES_DIR / "cross_source_leverage.png"))

pf_quarters = pd.to_datetime(pf_hf["quarter"]).dt.to_period("Q")
z1_quarters = df.index.to_period("Q")
overlap = pf_quarters.isin(z1_quarters).sum()
print(f"\nCross-source overlap: {overlap} quarters where both Z.1 and Form PF have data")
print(f"  Z.1 coverage:     {z1_quarters.min()} to {z1_quarters.max()}")
print(f"  Form PF coverage: {pf_quarters.min()} to {pf_quarters.max()}")


Cross-source overlap: 49 quarters where both Z.1 and Form PF have data
  Z.1 coverage:     2012Q4 to 2025Q3
  Form PF coverage: 2013Q1 to 2025Q1


## 6. Sidebar: GameStop Event in Aggregate Data

> **Important caveats:** The analysis below uses **industry-aggregate** Z.1 data (~1,000+ funds). Individual fund impacts (e.g., Melvin Capital's ~53% January 2021 loss) are diluted across the sector. Short positions — central to the GameStop story — are **not captured** in either 13F filings or Z.1 balance sheets. This section shows what aggregate data can and cannot reveal about a fund-level event.

In January 2021, retail investors on r/WallStreetBets triggered a massive short squeeze on GameStop (GME). The before/after comparison below shows industry-wide metrics, not individual fund exposure.

In [28]:
# GameStop window analysis — before and after Q1 2021
print("NOTE: These are industry aggregates across ~1,000+ funds.")
print("Individual fund impacts (e.g., Melvin Capital) are diluted.")
print("Short positions are NOT captured in Z.1 or 13F data.\n")

gme_date = pd.Timestamp("2021-03-31")

# Check if we have data around the GameStop event
if gme_date in df.index or df.index.max() >= gme_date:
    # Find the closest quarters
    pre_gme = df.loc[:gme_date].iloc[-4:]  # 4 quarters before
    post_gme = df.loc[gme_date:].iloc[:4]  # 4 quarters after (including Q1 2021)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    metrics = [
        ("leverage_ratio", "Leverage Ratio", "#c0392b"),
        ("equity_pct", "Equity Allocation (%)", "#2980b9"),
        ("derivative_to_assets", "Derivatives / Assets (%)", "#8e44ad"),
        ("Total net assets", "Net Assets ($B)", "#27ae60"),
    ]

    for ax, (col, title, color) in zip(axes.flat, metrics):
        # Full time series
        vals = df[col] * 100 if "pct" in col or "to_assets" in col else df[col]
        ax.plot(df.index, vals, linewidth=1.5, color=color, alpha=0.6)

        # Highlight GameStop window
        window_start = pd.Timestamp("2020-06-30")
        window_end = pd.Timestamp("2021-12-31")
        window = df.loc[window_start:window_end]
        window_vals = window[col] * 100 if "pct" in col or "to_assets" in col else window[col]
        ax.plot(window.index, window_vals, linewidth=3, color=color)

        ax.axvline(gme_date, color="red", linestyle="--", linewidth=1.5, alpha=0.7)
        ax.text(gme_date, ax.get_ylim()[1] * 0.95, "GME\nSqueeze", ha="center", fontsize=9, color="red", style="italic")
        ax.set_title(title)

    plt.suptitle("Hedge Fund Industry Metrics — GameStop Window (Aggregate Data)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Print before/after comparison
    if len(pre_gme) > 0 and len(post_gme) > 0:
        print("\n" + "=" * 60)
        print("BEFORE vs AFTER GameStop (4-Quarter Averages, Industry Aggregate)")
        print("=" * 60)
        for col, label in [
            ("leverage_ratio", "Leverage Ratio"),
            ("equity_pct", "Equity Allocation"),
            ("derivative_to_assets", "Derivatives/Assets"),
            ("Total net assets", "Net Assets ($B)"),
        ]:
            before = pre_gme[col].mean()
            after = post_gme[col].mean()
            pct_change = (after - before) / before * 100
            if "pct" in col or "to_assets" in col:
                print(f"  {label:25s}: {before:.1%} → {after:.1%}  ({pct_change:+.1f}%)")
            elif "ratio" in col:
                print(f"  {label:25s}: {before:.2f}x → {after:.2f}x ({pct_change:+.1f}%)")
            else:
                print(f"  {label:25s}: ${before:,.0f}B → ${after:,.0f}B ({pct_change:+.1f}%)")
else:
    print(f"Data ends at {df.index.max().date()} — does not cover GameStop (Q1 2021)")
    print("Showing the trajectory leading into the event instead...")

    # Show last 8 quarters of available data
    recent = df.iloc[-8:]
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(recent.index, recent["leverage_ratio"], "o-", linewidth=2, markersize=8, color="#c0392b")
    for idx, val in recent["leverage_ratio"].items():
        ax.annotate(f"{val:.3f}", (idx, val), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
    ax.set_title("Leverage Ratio — Trajectory Leading into GameStop (Jan 2021)")
    ax.set_ylabel("Leverage Ratio")
    plt.tight_layout()
    plt.show()

NOTE: These are industry aggregates across ~1,000+ funds.
Individual fund impacts (e.g., Melvin Capital) are diluted.
Short positions are NOT captured in Z.1 or 13F data.




BEFORE vs AFTER GameStop (4-Quarter Averages, Industry Aggregate)
  Leverage Ratio           : 0.42x → 0.42x (-0.5%)
  Equity Allocation        : 41.4% → 43.7%  (+5.5%)
  Derivatives/Assets       : 44.3% → 42.0%  (-5.1%)
  Net Assets ($B)          : $1,701B → $1,884B (+10.7%)


## 7. Statistical Analysis

In [29]:
# Time series decomposition of total assets
from statsmodels.tsa.seasonal import seasonal_decompose

if len(df) >= 8:
    total_assets_clean = df["Total assets"].dropna()
    decomp = seasonal_decompose(total_assets_clean, model="additive", period=4)

    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

    axes[0].plot(decomp.observed, color="#2c3e50", linewidth=1.5)
    axes[0].set_ylabel("Observed")
    axes[0].set_title("Time Series Decomposition — Hedge Fund Total Assets")

    axes[1].plot(decomp.trend, color="#2980b9", linewidth=2)
    axes[1].set_ylabel("Trend")

    axes[2].plot(decomp.seasonal, color="#27ae60", linewidth=1.5)
    axes[2].set_ylabel("Seasonal")

    axes[3].plot(decomp.resid, color="#e74c3c", linewidth=1, alpha=0.7)
    axes[3].scatter(decomp.resid.index, decomp.resid, color="#e74c3c", s=20, alpha=0.5)
    axes[3].axhline(0, color="gray", linewidth=0.5)
    axes[3].set_ylabel("Residual")

    for ax in axes:
        add_event_annotations(ax, ypos_frac=0.9)

    plt.tight_layout()
    plt.show()
else:
    print("Not enough data for seasonal decomposition")

In [30]:
# Rolling correlations — leverage vs equity allocation, leverage vs VIX
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Leverage vs equity allocation (rolling 8-quarter correlation)
rolling_corr_eq = df["leverage_ratio"].rolling(8).corr(df["equity_pct"])
axes[0].plot(df.index, rolling_corr_eq, linewidth=2, color="#2980b9")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].fill_between(df.index, rolling_corr_eq, 0, alpha=0.1, color="#2980b9")
axes[0].set_title("Rolling Correlation: Leverage vs. Equity Allocation (8Q)")
axes[0].set_ylabel("Correlation")
axes[0].set_ylim(-1, 1)
add_event_annotations(axes[0])

# Leverage vs VIX (if available)
if "VIX_mean" in df_merged.columns:
    rolling_corr_vix = df_merged["leverage_ratio"].rolling(8).corr(df_merged["VIX_mean"])
    axes[1].plot(df_merged.index, rolling_corr_vix, linewidth=2, color="#8e44ad")
    axes[1].axhline(0, color="gray", linewidth=0.5)
    axes[1].fill_between(df_merged.index, rolling_corr_vix, 0, alpha=0.1, color="#8e44ad")
    axes[1].set_title("Rolling Correlation: Leverage vs. VIX (8Q)")
    axes[1].set_ylabel("Correlation")
    axes[1].set_ylim(-1, 1)
    add_event_annotations(axes[1])
else:
    axes[1].text(0.5, 0.5, "VIX data not available", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [31]:
# Structural break detection — CUSUM on total assets growth
growth = df["total_assets_qoq"].dropna()

fig, ax = plt.subplots(figsize=(14, 5))

cumsum = np.cumsum(growth.values - growth.values.mean())
dates = growth.index

ax.plot(dates, cumsum, linewidth=2, color="#2c3e50")
ax.fill_between(dates, cumsum, 0, alpha=0.1, color="#2c3e50")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("CUSUM Chart — Total Assets QoQ Growth (Structural Break Detection)")
ax.set_ylabel("Cumulative Sum of Deviations")
add_event_annotations(ax)

# Identify potential break points
for i in range(2, len(cumsum) - 2):
    if (cumsum[i] > cumsum[i - 1] and cumsum[i] > cumsum[i + 1] and abs(cumsum[i]) > np.std(cumsum)) or (
        cumsum[i] < cumsum[i - 1] and cumsum[i] < cumsum[i + 1] and abs(cumsum[i]) > np.std(cumsum)
    ):
        ax.axvline(dates[i], color="orange", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

In [32]:
# Generate summary statistics for conclusions
print("=" * 70)
print("HEDGE FUND INDUSTRY ANALYSIS — KEY FINDINGS")
print("=" * 70)

first_q = f"{df.index.min().year} Q{df.index.min().quarter}"
last_q = f"{df.index.max().year} Q{df.index.max().quarter}"
print(f"\nPeriod: {first_q} to {last_q} ({len(df)} quarters)")

print("\n1. INDUSTRY GROWTH")
first_assets = df["Total assets"].iloc[0]
last_assets = df["Total assets"].iloc[-1]
years = len(df) / 4
if first_assets > 0:
    cagr = (last_assets / first_assets) ** (1 / years) - 1
    print(f"   Total assets grew from ${first_assets:,.0f}B to ${last_assets:,.0f}B")
    print(f"   CAGR: {cagr:.1%}")
else:
    print(f"   Total assets: ${last_assets:,.0f}B (latest)")
    print("   CAGR: N/A (no valid starting value)")

print("\n2. LEVERAGE TRENDS")
print(f"   Average leverage ratio: {df['leverage_ratio'].mean():.2f}x")
print(f"   Peak leverage: {df['leverage_ratio'].max():.2f}x ({df['leverage_ratio'].idxmax().date()})")
print(f"   Min leverage: {df['leverage_ratio'].min():.2f}x ({df['leverage_ratio'].idxmin().date()})")

print("\n3. ASSET ALLOCATION SHIFTS")
first_eq = df["equity_pct"].iloc[0]
last_eq = df["equity_pct"].iloc[-1]
print(f"   Equity allocation: {first_eq:.1%} → {last_eq:.1%}")
first_debt = df["debt_securities_pct"].iloc[0]
last_debt = df["debt_securities_pct"].iloc[-1]
print(f"   Debt securities:   {first_debt:.1%} → {last_debt:.1%}")

print("\n4. DERIVATIVE EXPOSURE")
print(f"   Average derivatives/assets: {df['derivative_to_assets'].mean():.1%}")
print(f"   Peak: {df['derivative_to_assets'].max():.1%} ({df['derivative_to_assets'].idxmax().date()})")

print("\n5. BORROWING PATTERNS")
print(
    f"   Foreign borrowing share: {df['foreign_borrowing_share'].iloc[0]:.1%} → {df['foreign_borrowing_share'].iloc[-1]:.1%}"
)
print(f"   Prime brokerage dominance: {df['prime_brokerage_pct'].mean():.1%} of total loans (avg)")

if not df_13f.empty and "value_usd" in df_13f.columns:
    latest_q = (
        df_13f["report_period"].dropna().astype(str).max() if "report_period" in df_13f.columns else "all filings"
    )
    df_snapshot = (
        df_13f[df_13f["report_period"].astype(str) == latest_q] if "report_period" in df_13f.columns else df_13f
    )
    df_eq = df_snapshot[df_snapshot["put_call"].fillna("").eq("")]
    print(f"\n6. 13F HOLDINGS — {latest_q} snapshot ({df_eq['fund'].nunique()} funds)")
    print(f"   Long equity/ETF positions: {len(df_eq):,}")
    top3 = df_eq.groupby("issuer")["value_usd"].sum().sort_values(ascending=False).head(3)
    print(f"   Top 3 long holdings: {', '.join(top3.index)}")

HEDGE FUND INDUSTRY ANALYSIS — KEY FINDINGS

Period: 2012 Q4 to 2025 Q3 (52 quarters)

1. INDUSTRY GROWTH
   Total assets grew from $1,145B to $3,258B
   CAGR: 8.4%

2. LEVERAGE TRENDS
   Average leverage ratio: 0.43x
   Peak leverage: 0.48x (2025-07-01)
   Min leverage: 0.40x (2017-04-01)

3. ASSET ALLOCATION SHIFTS
   Equity allocation: 33.9% → 42.0%
   Debt securities:   34.0% → 26.0%

4. DERIVATIVE EXPOSURE
   Average derivatives/assets: 56.2%
   Peak: 152.9% (2018-01-01)

5. BORROWING PATTERNS
   Foreign borrowing share: 14.8% → 16.9%
   Prime brokerage dominance: 78.3% of total loans (avg)

6. 13F HOLDINGS — 2025Q4 snapshot (8 funds)
   Long equity/ETF positions: 43,482
   Top 3 long holdings: ISHARES TR, NVIDIA CORPORATION, SPDR S&P 500 ETF TR


### Limitations

- **Aggregate vs. fund-level**: The Fed Z.1 data represents the entire domestic hedge fund sector (~1,000+ funds) — it masks wide variation across individual funds, strategies, and sizes. Form PF provides fund-level data but only for large advisers (>$1.5B AUM)
- **13F filing delays**: Holdings are reported ~45 days after quarter-end and only show long equity positions. Short positions are not disclosed
- **Survivorship bias**: Funds that close (like Melvin Capital) disappear from the aggregate data
- **Reporting thresholds**: Only institutional investment managers with >$100M in qualifying assets must file 13F reports
- **Cross-source timing**: Z.1 uses quarter-start dates, Form PF uses quarter labels, CFTC swaps are weekly, FCM data is monthly — alignment requires period-based joins that may introduce small timing mismatches

### Future Directions

- Add short-interest data (e.g., FINRA) to capture the short positioning invisible in 13F and Z.1
- Incorporate options flow data to reveal synthetic short exposure via puts
- Track hedge fund launches and closures to measure industry churn and survivorship effects
- Extend cross-source analysis to compare hedge fund leverage with broader financial sector leverage (banks, broker-dealers)
- Add non-U.S. hedge fund data for global industry comparison